# Set Up

In [ ]:
import os

if not os.path.exists('/content/repository'):
    !git clone https://github.com/annajli/cs6501

%cd cs6501

Cloning into 'cs6501'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 24 (delta 3), reused 24 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 2.27 MiB | 5.63 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/cs6501


In [ ]:
!git config --global user.email "tpb3rw@virginia.edu"
!git config --global user.name "Anna Li"

from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git remote set-url origin https://{token}@github.com/username/repository.git

In [ ]:
!mkdir topic6
%cd topic6

/content/cs6501/topic6


In [ ]:
# 0. Install zstd dependency
!apt-get install -y zstd

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background
import subprocess
subprocess.Popen(["ollama", "serve"])

# 3. Wait a moment for the server to initialize
import time
time.sleep(3)

# 4. Now you can pull the model
!ollama pull llava

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (849 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [ ]:
pip install langchain-ollama

# Exercise 1: Vision-Language LangGraph Chat Agent

Write a chat agent that can carry on a multi-turn conversation about an image you upload.  Use all that you have learned about good LangGraph style to structure the agent and manage the context.  If the program runs slowly, check if it helps to reduce the resolution of the uploaded image.


In [ ]:
import base64
import io
from typing import Annotated, TypedDict, List
from PIL import Image

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── State ──────────────────────────────────────────────────────────────────────

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    image_b64: str | None  # base64-encoded image (persists across turns)

# ── LLM ────────────────────────────────────────────────────────────────────────

llm = ChatOllama(model="llava", num_predict=256)

# ── Node: chat ─────────────────────────────────────────────────────────────────

def chat_node(state: AgentState) -> AgentState:
    """Send messages (with image on first turn) to the VLM."""
    system = SystemMessage(content=(
        "You are a helpful assistant that can analyze images and answer questions about them. "
        "Refer back to the uploaded image when answering follow-up questions."
    ))

    history = list(state["messages"])
    image_b64 = state.get("image_b64")

    if image_b64:
        # LangChain ChatOllama requires image_url format with a data URI
        last_msg = history[-1]
        text_content = last_msg.content if isinstance(last_msg.content, str) else ""
        history[-1] = HumanMessage(content=[
            {
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{image_b64}",
            },
            {"type": "text", "text": text_content},
        ])

    response = llm.invoke([system] + history)
    return {"messages": [response]}

# ── Graph ──────────────────────────────────────────────────────────────────────

graph_builder = StateGraph(AgentState)
graph_builder.add_node("chat", chat_node)
graph_builder.set_entry_point("chat")
graph_builder.add_edge("chat", END)
graph = graph_builder.compile()

print("✅ LangGraph agent compiled.")

✅ LangGraph agent compiled.


In [ ]:
# ── State management ───────────────────────────────────────────────────────────
conversation_state: AgentState = {"messages": [], "image_b64": None}

# ── Widgets ────────────────────────────────────────────────────────────────────
upload_btn   = widgets.FileUpload(accept="image/*", multiple=False, description="Upload Image")
text_input   = widgets.Text(placeholder="Ask a question about the image...", layout=widgets.Layout(width="70%"))
send_btn     = widgets.Button(description="Send", button_style="primary")
clear_btn    = widgets.Button(description="Clear Chat", button_style="warning")
image_out    = widgets.Output()
chat_out     = widgets.Output()
status_label = widgets.Label(value="Upload an image to begin.")

def resize_image(data: bytes, max_dim: int = 768) -> str:
    """Resize image and return base64 string (reduces latency)."""
    img = Image.open(io.BytesIO(data)).convert("RGB")
    w, h = img.size
    scale = min(max_dim / w, max_dim / h, 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def on_upload(change):
    global conversation_state
    if not upload_btn.value:
        return
    file_info = list(upload_btn.value.values())[0]
    img_bytes = file_info["content"]
    b64 = resize_image(img_bytes)
    conversation_state = {"messages": [], "image_b64": b64}
    with image_out:
        clear_output()
        display(Image.open(io.BytesIO(img_bytes)).resize((300, 300)))
    with chat_out:
        clear_output()
    status_label.value = f"✅ Image loaded ({file_info['metadata']['name']}). Ask your question!"

def on_send(b):
    global conversation_state
    user_text = text_input.value.strip()
    if not user_text:
        return
    if conversation_state["image_b64"] is None:
        status_label.value = "⚠️ Please upload an image first."
        return

    text_input.value = ""
    status_label.value = "🤔 Thinking..."

    with chat_out:
        print(f"You: {user_text}")

    # Add user message & run graph
    conversation_state["messages"].append(HumanMessage(content=user_text))
    result = graph.invoke(conversation_state)

    # Update state: keep image, accumulate messages
    conversation_state["messages"] = result["messages"]
    # After first turn, remove image from state to avoid re-sending on every turn
    # (keep it only for first message; comment out next line to always send image)
    # conversation_state["image_b64"] = None

    ai_response = result["messages"][-1].content
    with chat_out:
        print(f"Assistant: {ai_response}\n{'-'*60}")
    status_label.value = "Ready. Ask a follow-up question!"

def on_clear(b):
    global conversation_state
    conversation_state["messages"] = []
    with chat_out:
        clear_output()
    status_label.value = "Chat cleared. Ask a new question about the image."

upload_btn.observe(on_upload, names="value")
send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)  # Enter key sends

# ── Layout ─────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HTML("<h2>🖼️ VLM Chat Agent (Exercise 1)</h2>"),
    upload_btn,
    image_out,
    status_label,
    widgets.HBox([text_input, send_btn, clear_btn]),
    chat_out,
]))

ResponseError: model runner has unexpectedly stopped, this may be due to resource limitations or an internal error, check ollama server logs for details (status code: 500)